# 🛒 Intelligent Retail Shelf Monitoring — YOLO Model Training
**Computer Vision Capstone Project · ITA 0506**

Target Product Classes:
- `0: Coke`
- `1: Pepsi`
- `2: Sprite`

This notebook trains a custom lightweight **YOLOv8n** or **YOLO11n** object detection model on Google Colab with free GPU acceleration (T4 GPU).

### Complete Workflow:
1. Check GPU (`!nvidia-smi`)
2. Install Ultralytics & Dependencies
3. Set up Dataset (`data.yaml` or Roboflow / ZIP)
4. Verify Dataset counts and paths
5. Train YOLOv8n / YOLO11n (50 epochs, imgsz 640)
6. Validate Metrics (Precision, Recall, mAP@50, mAP@50-95)
7. Test on Unseen Shelf Images and Verify Counts (Coke, Pepsi, Sprite)
8. Download `best.pt` to place into `retail-shelf-monitoring/models/best.pt`

## Cell 1 — Check GPU Availability

In [ ]:
# ── Cell 1: Check GPU ───────────────────────────────────────────
!nvidia-smi

import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    print('⚠️ Running on CPU. Go to Runtime -> Change runtime type -> Select T4 GPU for 10x faster training.')

## Cell 2 — Install Ultralytics & Requirements

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────
!pip install --upgrade ultralytics roboflow opencv-python-headless matplotlib pyyaml -q
print('✅ Ultralytics YOLO installed successfully!')

## Cell 3 — Import Libraries

In [ ]:
# ── Cell 3: Imports ─────────────────────────────────────────────
import os
import shutil
import yaml
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
from google.colab import files

print('✅ Libraries loaded.')

## Cell 4 — Dataset Setup (Roboflow or Upload ZIP)
Choose either:
- **Option A:** Download directly from Roboflow via API key
- **Option B:** Upload a `dataset.zip` file containing `images/` and `labels/`

In [ ]:
# ── Cell 4A: Option A — Roboflow Download ────────────────────────
USE_ROBOFLOW = False  # Set to True if you have a Roboflow project

if USE_ROBOFLOW:
    from roboflow import Roboflow
    rf = Roboflow(api_key='YOUR_API_KEY')
    project = rf.workspace('YOUR_WORKSPACE').project('retail-shelf')
    version = project.version(1)
    dataset = version.download('yolov8')
    DATASET_DIR = Path(dataset.location)
    DATA_YAML = DATASET_DIR / 'data.yaml'
    print('Roboflow dataset downloaded to:', DATASET_DIR)
else:
    print('Using Option B (ZIP Upload / Colab Files). Proceed to Cell 4B.')

In [ ]:
# ── Cell 4B: Option B — Extract dataset.zip ──────────────────────
DATASET_DIR = Path('/content/dataset')
DATASET_DIR.mkdir(parents=True, exist_ok=True)

zip_files = list(Path('/content').glob('*.zip'))
if zip_files:
    target_zip = zip_files[0]
    print(f'Extracting {target_zip} to /content/dataset...')
    !unzip -q -o "{target_zip}" -d /content/dataset
    print('✅ Dataset extracted.')
else:
    print('Please upload your dataset ZIP file now (click Choose Files below):')
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith('.zip'):
            !unzip -q -o "{fname}" -d /content/dataset
            print(f'✅ Extracted {fname} to /content/dataset')

## Cell 5 — Verify Dataset & Configure `dataset.yaml`

In [ ]:
# ── Cell 5: Create and verify dataset.yaml ───────────────────────
yaml_content = '''
path: /content/dataset
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: Coke
  1: Pepsi
  2: Sprite
'''

DATA_YAML = Path('/content/dataset/dataset.yaml')
DATA_YAML.parent.mkdir(parents=True, exist_ok=True)
with open(DATA_YAML, 'w') as f:
    f.write(yaml_content.strip())

print('✅ dataset.yaml written successfully.')

# Check split image and label counts
for split in ['train', 'val', 'test']:
    img_path = Path('/content/dataset/images') / split
    lbl_path = Path('/content/dataset/labels') / split
    num_imgs = len(list(img_path.glob('*.*'))) if img_path.exists() else 0
    num_lbls = len(list(lbl_path.glob('*.txt'))) if lbl_path.exists() else 0
    print(f'  {split.upper():5s} -> Images: {num_imgs} | Labels: {num_lbls}')

## Cell 6 & 7 — Train YOLOv8n / YOLO11n Model (50 Epochs)

In [ ]:
# ── Cell 6: Train Model ─────────────────────────────────────────
# Model selection: 'yolov8n.pt' or 'yolo11n.pt'
BASE_MODEL = 'yolov8n.pt'
print('Initializing model:', BASE_MODEL)
model = YOLO(BASE_MODEL)

# Start fine-tuning
results = model.train(
    data     = str(DATA_YAML),
    epochs   = 50,          # 50 epochs standard for solid mAP
    imgsz    = 640,
    batch    = 16,          # change to 8 if GPU out of memory
    patience = 10,          # early stopping patience
    workers  = 2,
    project  = '/content/runs',
    name     = 'retail_shelf',
    exist_ok = True,
    device   = 0 if torch.cuda.is_available() else 'cpu',
    verbose  = True,
)

print('✅ Training finished successfully!')

## Cell 8 — Validate Model & View Evaluation Curves

In [ ]:
# ── Cell 8: Validation ──────────────────────────────────────────
best_weights = Path('/content/runs/retail_shelf/weights/best.pt')
trained_model = YOLO(str(best_weights))

metrics = trained_model.val(data=str(DATA_YAML), imgsz=640, verbose=True)

print('\n' + '='*48)
print('     FINAL MODEL PERFORMANCE METRICS')
print('='*48)
print(f'  Precision (mp)       : {metrics.box.mp:.4f}')
print(f'  Recall (mr)          : {metrics.box.mr:.4f}')
print(f'  mAP @ 0.50 (map50)   : {metrics.box.map50:.4f}')
print(f'  mAP @ 0.50:0.95 (map): {metrics.box.map:.4f}')
print('='*48)

# Display results curve
results_img = Path('/content/runs/retail_shelf/results.png')
if results_img.exists():
    plt.figure(figsize=(12, 6))
    plt.imshow(cv2.cvtColor(cv2.imread(str(results_img)), cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Training and Validation Curves')
    plt.show()

## Cell 9 — Test Model on Unseen Shelf Images & Verify Product Counts

In [ ]:
# ── Cell 9: Inference & Product Counting ────────────────────────
test_imgs = list(Path('/content/dataset/images/test').glob('*.*'))
if not test_imgs:
    test_imgs = list(Path('/content/dataset/images/val').glob('*.*'))

if test_imgs:
    test_path = str(test_imgs[0])
    pred = trained_model.predict(source=test_path, conf=0.25, verbose=False)[0]
    
    # Product Counting logic required by Streamlit application
    product_counts = {'Coke': 0, 'Pepsi': 0, 'Sprite': 0}
    for c in pred.boxes.cls:
        cname = trained_model.names[int(c)]
        product_counts[cname] = product_counts.get(cname, 0) + 1
        
    print('\n📊 Extracted Shelf Quantities for Streamlit:')
    for prod, cnt in product_counts.items():
        status = 'Available' if cnt >= 5 else ('Low Stock' if cnt >= 1 else 'Out of Stock')
        print(f'   {prod:7s} : {cnt:2d} -> {status}')
        
    # Plot bounding boxes
    plotted = pred.plot()
    plt.figure(figsize=(9, 9))
    plt.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f'Detected: {product_counts}')
    plt.show()
else:
    print('No test images found.')

## Cell 10 — Export and Download `best.pt`

In [ ]:
# ── Cell 10: Export best.pt ─────────────────────────────────────
export_file = Path('/content/best.pt')
shutil.copy2(best_weights, export_file)
print(f'✅ Model copied to {export_file} ({export_file.stat().st_size / (1024*1024):.2f} MB)')

print('\n📥 Downloading best.pt to your local computer...')
files.download(str(export_file))

print('\n👉 Next Step:')
print('   Place this downloaded best.pt into your project folder:')
print('   retail-shelf-monitoring/models/best.pt')
print('   Then launch Streamlit: streamlit run app.py')